# Sesión 3.05 · Qdrant local mediante SDK nativo

Qdrant será una alternativa local de esta sesión. Lo ejecutaremos en Docker como un servicio independiente y el notebook actuará como cliente mediante su SDK nativo. Esa separación permite observar una arquitectura real: la aplicación no es la base de datos, sino que se comunica con ella mediante una API.

El laboratorio mantendrá exactamente el contrato común: utilizará los embeddings ya calculados, no delegará la inferencia en el proveedor y no añadirá LangChain. Si el ranking difiere del oráculo exacto, podremos investigar el esquema, el índice, los filtros, la visibilidad de las escrituras o la semántica de la respuesta sin mezclar esas causas con otro encoder o framework.

<a id="s03-qdrant-indice"></a>

## Índice de contenidos

1. [Configuración del entorno y del índice](#s03-qdrant-configuracion)
2. [Ingesta](#s03-qdrant-ingesta)
3. [Verificación de la carga](#s03-qdrant-verificacion)
4. [Resultados de búsqueda](#s03-qdrant-resultados)
5. [Búsqueda sin filtro](#s03-qdrant-busqueda-sin-filtro)
6. [Búsqueda filtrada](#s03-qdrant-busqueda-filtrada)
7. [Operaciones CRUD](#s03-qdrant-operaciones-crud)
8. [Informe de ejecución](#s03-qdrant-informe)
9. [Consideraciones específicas](#s03-qdrant-consideraciones)
10. [Próximos pasos](#s03-qdrant-proximos-pasos)
11. [Limpieza](#s03-qdrant-limpieza)

## Objetivo del laboratorio

También registraremos tiempos, pero solo para describir esta ejecución concreta. No los utilizaremos para comparar proveedores, porque una llamada a un servicio gestionado y una ejecución local no comparten red, hardware, caché ni condiciones de carga.

El recorrido comenzará con un **preflight**. Si faltan credenciales, el índice no existe o su configuración no coincide con el contrato esperado, el notebook se detendrá y mostrará la causa. Continuar después de un fallo con una variable vacía como `results = []` ocultaría información importante: no podríamos distinguir entre una búsqueda que realmente no encontró vecinos y una consulta que nunca llegó a ejecutarse.

In [1]:
from pathlib import Path
import json
import os
import platform
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vector_database_session import (
    ProviderRun,
    SearchHit,
    evaluate_run,
    exact_top_k,
    iter_record_batches,
    load_session_data,
    record_id_for_product,
    validate_resource_name,
    wait_until,
    write_provider_run,
)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / ".env")
data = load_session_data(memory_map=True)
TELEVISOR_QUERY_ID = "semantic-101352"
TALADRO_QUERY_ID = "semantic-100455"
TOP_K = 10

In [2]:
RESOURCE_NAME = validate_resource_name(os.getenv("QDRANT_COLLECTION", "bbdd-vectoriales-s03-qdrant"))
BATCH_SIZE = 500
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "esci-es-s03")
TEMPORARY_TEST_ID = record_id_for_product("S03-TEMPORARY-TEST")
assert os.getenv("S03_ALLOW_REMOTE_CLEANUP", "false").lower() != "true", "La limpieza remota no pertenece a la ejecución docente"
print({"python": platform.python_version(), "resource": RESOURCE_NAME, "batch_size": BATCH_SIZE})

{'python': '3.12.13', 'resource': 'bbdd-vectoriales-s03-qdrant', 'batch_size': 500}


<a id="s03-qdrant-configuracion"></a>

## 1. Configuración del entorno y del índice

Qdrant expone un modelo sencillo: colección, vector y payload. El vector tendrá 384 dimensiones y utilizará coseno; el payload contendrá el documento y los metadatos necesarios para interpretar y filtrar el resultado.

Para vectores densos, Qdrant utiliza HNSW como índice ANN. No podemos elegir una familia IVF, DiskANN o ScaNN equivalente a la de Milvus, pero sí fijar los parámetros relevantes: m=24, ef_construct=120 y hnsw_ef=128 en la consulta. Así podemos relacionar la exploración del grafo con el recall observado.

Qdrant también puede ejecutar una búsqueda exacta bajo petición y puede preferir fuerza bruta para subconjuntos pequeños o filtros muy selectivos. Esas rutas no convierten el motor en una colección de algoritmos ANN intercambiables: son decisiones del planificador o de una consulta concreta.


In [3]:
#NOTE: Ejecuta esta celda solo si no has levantado el contenedor de Weaviate antes.
!docker compose -f ../deploy/qdrant/compose.yaml up -d

[+] up 1/1
 ✔ Container bbdd-vectoriales-s03-qdrant Running                            0.0s


In [4]:
import importlib.metadata
import qdrant_client
from qdrant_client import QdrantClient, models

provider_version = importlib.metadata.version("qdrant-client")
url = os.getenv("QDRANT_URL", "http://localhost:6333")
api_key = os.getenv("QDRANT_API_KEY", "").strip() or None

client = QdrantClient(url=url, api_key=api_key, timeout=60)

existing = {item.name for item in client.get_collections().collections}
if RESOURCE_NAME not in existing:
    client.create_collection(
        collection_name=RESOURCE_NAME,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(m=24, ef_construct=120),
    )
    client.create_payload_index(
        collection_name=RESOURCE_NAME,
        field_name="metadata.brand",
        field_schema=models.PayloadSchemaType.KEYWORD,
        wait=True,
    )

collection_info = client.get_collection(RESOURCE_NAME)
vector_config = collection_info.config.params.vectors
assert vector_config.size == 384 and vector_config.distance == models.Distance.COSINE, vector_config
print({"version": provider_version, "target": url, "status": collection_info.status})

{'version': '1.18.0', 'target': 'http://localhost:6333', 'status': <CollectionStatus.GREEN: 'green'>}


Antes de ingerir, conviene revisar tanto la configuración del vector como el índice de payload sobre metadata.brand. Son estructuras diferentes: una guía la búsqueda por similitud y la otra ayuda a aplicar el filtro durante la recuperación.

Crear el índice de payload antes de cargar los datos también forma parte del experimento. Qdrant puede aprovecharlo al construir las conexiones adicionales que hacen compatible HNSW con filtros. Si cambiáramos el orden de esas operaciones, estaríamos modificando algo más que una línea de configuración.


<a id="s03-qdrant-ingesta"></a>

## 2. Ingesta

Cada producto utilizará el mismo UUIDv5 en los cinco motores. Como el identificador se genera de forma determinista a partir de `product_id`, volver a ejecutar la ingesta produce exactamente los mismos IDs.

La operación `upsert` aprovecha esa propiedad: si el registro no existe, lo inserta; si ya estaba presente, lo reemplaza o actualiza según la semántica del motor. Gracias a ello, el notebook puede reanudarse después de una interrupción sin crear duplicados.

Esto no significa que toda la tubería ofrezca una garantía de *exactly once*. Si el proceso falla a mitad de la carga, algunos batches pueden haber sido confirmados y otros no. La idempotencia no evita ese estado parcial, pero hace que repetir la operación sea seguro: los lotes ya escritos se procesarán de nuevo sobre los mismos IDs y los pendientes podrán completarse.

Utilizaremos batches de 500 registros como punto de partida para la práctica. No debe interpretarse como un tamaño óptimo universal. En producción, el valor adecuado depende de los límites de cada petición, la memoria disponible en el cliente, el tamaño de los metadatos, la latencia de red y el grado de concurrencia.

Registraremos el tiempo total de ingesta para describir esta ejecución y detectar posibles anomalías. No lo utilizaremos para comparar Pinecone con motores ejecutados en otros entornos, porque las condiciones de hardware, red y despliegue no son equivalentes.

In [5]:
ingestion_started = time.perf_counter()
for batch_number, records in enumerate(iter_record_batches(data, batch_size=BATCH_SIZE), start=1):
    client.upsert(
        collection_name=RESOURCE_NAME,
        wait=True,
        points=[
            models.PointStruct(id=record.record_id, vector=record.embedding, payload=record.qdrant_payload())
            for record in records
        ],
    )
    if batch_number % 20 == 0:
        print(f"{batch_number * BATCH_SIZE:,} puntos confirmados")
ingestion_ms = (time.perf_counter() - ingestion_started) * 1000

10,000 puntos confirmados
20,000 puntos confirmados
30,000 puntos confirmados
40,000 puntos confirmados
50,000 puntos confirmados


<a id="s03-qdrant-verificacion"></a>

## 3. Verificación de la carga

Que el servidor haya aceptado 50.000 escrituras no significa todavía que los 50.000 registros estén disponibles para búsqueda. La ingesta y la visibilidad forman parte de momentos distintos del ciclo de escritura.

Por eso no daremos por completada la carga a partir del número de batches enviados ni de las respuestas correctas del cliente. Consultaremos el estado que expone el propio motor y comprobaremos cuántos registros reconoce dentro del namespace.

En un servicio con visibilidad eventual, ese recuento puede tardar unos instantes en alcanzar el valor esperado. Esperaremos de forma acotada y registraremos la evolución observada. Si se agota el plazo sin llegar a 50.000 registros visibles, el notebook fallará mostrando el último estado recibido.

De este modo distinguiremos entre tres situaciones diferentes: una escritura rechazada, una escritura aceptada pero todavía no visible y una carga completamente disponible para consulta.

In [6]:
collection_info = client.get_collection(RESOURCE_NAME)
record_count = int(collection_info.points_count)
visibility_seconds, visibility_attempts = 0.0, 1

assert record_count == 50_000, f'Recuento inesperado: {record_count}'
print({'record_count': record_count, 'ingestion_ms': ingestion_ms, 'visible_s': visibility_seconds})

{'record_count': 50000, 'ingestion_ms': 15937.148666000041, 'visible_s': 0.0}


Un recuento correcto no prueba alineación de IDs, contenido del payload ni calidad del ranking. Sí descarta una clase importante de explicaciones: batches perdidos o namespace equivocado. La validación completa combina invariantes, búsquedas conocidas y mutaciones.

<a id="s03-qdrant-resultados"></a>

## 4. Resultados de búsqueda

Cada motor devuelve sus resultados con una estructura propia. Para poder comparar rankings y mostrar una tabla común, traduciremos esas respuestas a un modelo compartido llamado `SearchHit`.

Ese modelo conservará el ID recuperado, pero también tres campos necesarios para interpretar correctamente la puntuación: `native_score`, `score_kind` y `higher_is_better`. Así sabremos si el proveedor devuelve una similitud o una distancia y en qué sentido debe ordenarse.

La normalización no convierte esas puntuaciones en magnitudes equivalentes. Una distancia de 0,2 y una similitud de 0,8 pueden inducir el mismo ranking, pero no representan la misma cantidad ni deben compararse directamente entre motores.

El objetivo es más sencillo: evitar repetir cinco veces el código de evaluación y trabajar con una interfaz común sin ocultar la semántica original de cada proveedor.

In [7]:
def native_search(query_vector, *, k=10, brand=None):
    result_filter = None
    if brand:
        result_filter = models.Filter(
            must=[models.FieldCondition(key="metadata.brand", match=models.MatchValue(value=brand))]
        )
    response = client.query_points(
        collection_name=RESOURCE_NAME,
        query=np.asarray(query_vector, dtype=np.float32).tolist(),
        query_filter=result_filter,
        search_params=models.SearchParams(hnsw_ef=128, exact=False),
        limit=k,
        with_payload=True,
    ).points
    return [
        SearchHit(
            record_id=str(item.id),
            product_id=item.payload["metadata"]["product_id"],
            vector_id=int(item.payload["metadata"]["vector_id"]),
            title=item.payload["metadata"]["title"],
            brand=item.payload["metadata"].get("brand", ""),
            native_score=float(item.score),
            score_kind="similarity",
            higher_is_better=True,
            rank=rank,
        )
        for rank, item in enumerate(response, start=1)
    ]

<a id="s03-qdrant-busqueda-sin-filtro"></a>

## 5. Búsqueda sin filtro

Volveremos ahora a la consulta problemática del televisor. Antes de inspeccionar la tabla, conviene formular una predicción: si Qdrant reproduce fielmente el espacio generado por E5, debería devolver en primera posición el mismo producto que el oráculo exacto.

En este caso, ese producto es un mantel. El resultado es claramente poco útil para la intención de búsqueda, pero su presencia no demuestra un fallo de la base de datos. Al contrario: si el motor devuelve el mismo UUID y mantiene un `recall@10` alto frente al oráculo, estará reproduciendo correctamente una geometría semántica que ya contenía ese error.

Que cinco bases de datos distintas recuperen el mismo mantel no lo convierte en relevante. Lo convierte en una evidencia reproducible de que el problema se encuentra antes, en el encoder, en la representación del texto o en los datos con los que se construyó el espacio.

Esta distinción será central durante la inspección. La fidelidad mide cuánto respeta el motor el ranking definido por los vectores; la relevancia mide si ese ranking responde realmente a la necesidad del usuario. Ambas propiedades pueden coincidir, pero no son equivalentes.

In [8]:
television_row = data.query_row(TELEVISOR_QUERY_ID)
television_vector = data.query_vector(TELEVISOR_QUERY_ID)

exact_hits = exact_top_k(data, television_vector, k=TOP_K)

query_started = time.perf_counter()
native_hits = native_search(television_vector, k=TOP_K)
query_ms = (time.perf_counter() - query_started) * 1000

television_evaluation = evaluate_run(exact_hits, native_hits, k=TOP_K)

display(Markdown(f"**Consulta:** {television_row['query_text']}"))
display(pd.DataFrame([hit.as_dict() for hit in native_hits]))
television_evaluation

**Consulta:** busco un televisor pequeño de unas setenta centímetros para la cocina

,record_id,product_id,vector_id,title,brand,native_score,score_kind,higher_is_better,rank
0,95364a06-f359-5305-8d9e-b11893642cd3,B075Q9F6J1,16938,"Mantel de tela de algodón y lino de TJW, color...",TJW,0.884411,similarity,True,1
1,2cb8987c-0d14-57c7-9cc6-fea36644b1cc,B08KD2RX1J,42059,"SONGMICS Mueble de TV, Armario de TV, Mesa de ...",SONGMICS,0.873443,similarity,True,2
2,eb30b55f-b66c-5bf8-954f-b4f8b798b3ea,B08T7YZDQ9,44349,TV de red inteligente LED de 32 pulgadas / 42 ...,household items,0.870914,similarity,True,3
3,186c65d6-83c5-51a5-a105-a91a84e37fd6,B08TLXPSPB,44443,"Home appliances Smart TV 4K UHD con WiFi, Tele...",Home appliances,0.869468,similarity,True,4
4,faeae86e-2915-5eb9-959a-db8f4e63674a,B00XOZ1UIY,8658,"SoBuy FRG092-W,Soporte para microondas, Estant...",SoBuy,0.868736,similarity,True,5
5,64d50f61-9631-57f5-9504-5cfdaa17a151,B08TM15S1N,44444,"Home appliances Televisores Smart 4K UHD TV, T...",Home appliances,0.868338,similarity,True,6
6,bef55ecb-033f-5ffd-9951-6ca8a67e7aa8,B07X1XP82C,33911,"Nishore Mesa para TV con 1 Cajón, 1 Estante y ...",Nishore,0.868328,similarity,True,7
7,2bc62f72-ba1b-51c7-8117-738c81520898,B07MM4539M,26780,BONTEC Soporte TV Pie TV Peanas Giratorio Sopo...,BONTEC,0.868226,similarity,True,8
8,89f6dbf7-9305-5b13-888a-20f48bb63ac8,B075NRXVG8,16917,"5 PCS barras extensibles ajustable de 11,8 pul...",HAOYUNTE,0.868041,similarity,True,9
9,761cf124-42cd-58dc-8b53-f24d5fbea629,B07Q4GH7D5,29123,"Schneider Consumer - Televisión LED 32"" LED32-...",SCHNEIDER,0.867816,similarity,True,10


{'k': 10,
 'exact_record_ids': ['95364a06-f359-5305-8d9e-b11893642cd3',
  '2cb8987c-0d14-57c7-9cc6-fea36644b1cc',
  'eb30b55f-b66c-5bf8-954f-b4f8b798b3ea',
  '186c65d6-83c5-51a5-a105-a91a84e37fd6',
  'faeae86e-2915-5eb9-959a-db8f4e63674a',
  '64d50f61-9631-57f5-9504-5cfdaa17a151',
  'bef55ecb-033f-5ffd-9951-6ca8a67e7aa8',
  '2bc62f72-ba1b-51c7-8117-738c81520898',
  '89f6dbf7-9305-5b13-888a-20f48bb63ac8',
  '761cf124-42cd-58dc-8b53-f24d5fbea629'],
 'provider_record_ids': ['95364a06-f359-5305-8d9e-b11893642cd3',
  '2cb8987c-0d14-57c7-9cc6-fea36644b1cc',
  'eb30b55f-b66c-5bf8-954f-b4f8b798b3ea',
  '186c65d6-83c5-51a5-a105-a91a84e37fd6',
  'faeae86e-2915-5eb9-959a-db8f4e63674a',
  '64d50f61-9631-57f5-9504-5cfdaa17a151',
  'bef55ecb-033f-5ffd-9951-6ca8a67e7aa8',
  '2bc62f72-ba1b-51c7-8117-738c81520898',
  '89f6dbf7-9305-5b13-888a-20f48bb63ac8',
  '761cf124-42cd-58dc-8b53-f24d5fbea629'],
 'overlap_record_ids': ['95364a06-f359-5305-8d9e-b11893642cd3',
  '2cb8987c-0d14-57c7-9cc6-fea36644b1cc',

Interpreta `recall@10` como una medida de fidelidad algorítmica frente al oráculo exacto, no como una medida de relevancia humana. Un valor inferior a uno indica que el motor no ha reproducido por completo el top-10 de fuerza bruta, pero no identifica por sí solo la causa. La pérdida puede proceder del índice ANN, de una configuración de búsqueda demasiado restrictiva o de que algunos registros todavía no sean visibles para la consulta.

Un `recall@10` igual a uno cuenta una historia distinta. Si el ranking coincide con el oráculo y el mantel sigue apareciendo en primera posición, la base de datos ha reproducido correctamente el espacio de E5. En ese caso, el error debe atribuirse a la representación o al modelo, no al mecanismo de recuperación.

> **Pregunta.** Para distinguir una pérdida ANN estable de una escritura todavía no visible, repetiría la misma consulta después de verificar por ID y por recuento que todos los registros esperados están disponibles. Si el recall mejora a medida que se completa la visibilidad, el problema era de indexación o consistencia. Si permanece estable una vez confirmado el snapshot completo, la pérdida apunta al ANN o a su configuración.

<a id="s03-qdrant-busqueda-filtrada"></a>

## 6. Búsqueda filtrada

Ejecutaremos la misma consulta sobre taladros de dos formas. La primera buscará los vecinos más próximos dentro de todo el catálogo. La segunda añadirá la condición `brand == "Einhell"` directamente a la petición enviada al motor.

Esto significa que el top-$k$ filtrado debe calcularse sobre el conjunto de productos Einhell, no sobre los diez primeros resultados de la búsqueda global. No recuperaremos primero diez vecinos y eliminaremos después los que pertenezcan a otras marcas, porque ese procedimiento podría devolver menos resultados y perder candidatos válidos situados más abajo en el ranking general.

La comparación permitirá observar cómo cambia el universo de búsqueda cuando el filtro forma parte del contrato del motor. También compararemos los IDs devueltos con un oráculo exacto construido sobre el mismo subconjunto de productos Einhell, de modo que podamos medir la fidelidad del ranking condicionado y no solo comprobar que todos los resultados pertenecen a la marca solicitada.

In [9]:
drill_row = data.query_row(TALADRO_QUERY_ID)
drill_vector = data.query_vector(TALADRO_QUERY_ID)

global_drill_hits = native_search(drill_vector, k=TOP_K)
filtered_drill_hits = native_search(drill_vector, k=TOP_K, brand="Einhell")

exact_filtered_hits = exact_top_k(data, drill_vector, k=TOP_K, brand="Einhell")

filtered_evaluation = evaluate_run(exact_filtered_hits, filtered_drill_hits, k=TOP_K)

assert filtered_drill_hits and all(hit.brand == "Einhell" for hit in filtered_drill_hits)
display(Markdown(f"**Consulta:** {drill_row['query_text']}"))
display(pd.DataFrame([hit.as_dict() for hit in global_drill_hits]))
display(pd.DataFrame([hit.as_dict() for hit in filtered_drill_hits]))
filtered_evaluation

**Consulta:** quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe

,record_id,product_id,vector_id,title,brand,native_score,score_kind,higher_is_better,rank
0,e05576b6-795d-57c2-8896-96bb2562f148,B01N6Y6G16,13627,Einhell Atornillador inalámbrico TE-CD 18 Li-i...,Einhell,0.890112,similarity,True,1
1,c3189663-9488-5843-a713-0adece2b7c47,B01AB2KU5G,10098,Einhell Expert Martillo perforador y cincelado...,Einhell,0.883580,similarity,True,2
2,38e4bcab-8809-59de-815f-e0d99311309a,B08ZMZZRJH,45746,Einhell Kit Martillo perforador con batería HE...,Einhell,0.882192,similarity,True,3
3,59263e4f-021c-5bbf-8cde-6d1f1c3dd406,B07GSD93Q8,23298,Einhell Taladro de impacto sin cable TE-CD 12/...,Einhell,0.877822,similarity,True,4
4,726c2fe3-5608-5d6a-aad0-aaed494e4e1e,B00IYEEY0Q,6476,"Einhell Martillo perforador TC-RH 900 (900 W, ...",Einhell,0.876949,similarity,True,5
5,1be0e60a-b1bb-5c80-a1b6-af16fa87cbb3,B07XFFZFDR,34289,GREENCUT TD210L - Taladro atornillador perfora...,Greencut,0.873279,similarity,True,6
6,ab8f49aa-eee8-5afc-838f-058394f7b4be,B07N4K27PY,27311,Ryobi R18PD7-220B Taladro Percutor sin escobil...,Ryobi,0.872409,similarity,True,7
7,1bb3c7bd-6f68-5c63-ba54-9247f40e6c50,B093P4L7G9,46725,KATSU FIT-BAT 21V Motor inalámbrico sin escobi...,KATSU Tools,0.871594,similarity,True,8
8,882f4f74-4779-5887-a239-7d19e4245e78,B01N5T6SL4,13564,BLACK+DECKER BL188KB-QW - Taladro Percutor Mot...,Black+Decker,0.871185,similarity,True,9
9,2389fde0-c2e1-535d-a909-ba7e0b1aacb1,B0798C8QF6,19028,Einhell Herramienta multifuncional TC-MG 220/1...,Einhell,0.869544,similarity,True,10


,record_id,product_id,vector_id,title,brand,native_score,score_kind,higher_is_better,rank
0,e05576b6-795d-57c2-8896-96bb2562f148,B01N6Y6G16,13627,Einhell Atornillador inalámbrico TE-CD 18 Li-i...,Einhell,0.890112,similarity,True,1
1,c3189663-9488-5843-a713-0adece2b7c47,B01AB2KU5G,10098,Einhell Expert Martillo perforador y cincelado...,Einhell,0.883580,similarity,True,2
2,38e4bcab-8809-59de-815f-e0d99311309a,B08ZMZZRJH,45746,Einhell Kit Martillo perforador con batería HE...,Einhell,0.882192,similarity,True,3
3,59263e4f-021c-5bbf-8cde-6d1f1c3dd406,B07GSD93Q8,23298,Einhell Taladro de impacto sin cable TE-CD 12/...,Einhell,0.877822,similarity,True,4
4,726c2fe3-5608-5d6a-aad0-aaed494e4e1e,B00IYEEY0Q,6476,"Einhell Martillo perforador TC-RH 900 (900 W, ...",Einhell,0.876949,similarity,True,5
5,2389fde0-c2e1-535d-a909-ba7e0b1aacb1,B0798C8QF6,19028,Einhell Herramienta multifuncional TC-MG 220/1...,Einhell,0.869544,similarity,True,6
6,45186858-dd6f-595c-af22-ce4eda3ab7b6,B01MYUJ1A8,13221,Einhell Cepillo eléctrico con cable - TC-PL 75...,Einhell,0.855668,similarity,True,7
7,335c7af8-81f8-5fdd-8146-08ee17861680,B01BMABAXM,10357,Einhell 4259950 CC-IW 950 Llave de impacto (95...,Einhell,0.851736,similarity,True,8
8,7f535f14-3faf-57d4-bff2-45effbadf1db,B00G66VIOY,5964,"Einhell Lijadora Delta TC-DS 19 (190W, 20000 r...",Einhell,0.851301,similarity,True,9
9,29c0473b-46f9-5273-bab6-5f5b56dbcfc0,B00HWS8VRC,6232,Einhell Bomba de pozo profundo GC-DW 1300 N (1...,Einhell,0.850155,similarity,True,10


{'k': 10,
 'exact_record_ids': ['e05576b6-795d-57c2-8896-96bb2562f148',
  'c3189663-9488-5843-a713-0adece2b7c47',
  '38e4bcab-8809-59de-815f-e0d99311309a',
  '59263e4f-021c-5bbf-8cde-6d1f1c3dd406',
  '726c2fe3-5608-5d6a-aad0-aaed494e4e1e',
  '2389fde0-c2e1-535d-a909-ba7e0b1aacb1',
  '45186858-dd6f-595c-af22-ce4eda3ab7b6',
  '335c7af8-81f8-5fdd-8146-08ee17861680',
  '7f535f14-3faf-57d4-bff2-45effbadf1db',
  '29c0473b-46f9-5273-bab6-5f5b56dbcfc0'],
 'provider_record_ids': ['e05576b6-795d-57c2-8896-96bb2562f148',
  'c3189663-9488-5843-a713-0adece2b7c47',
  '38e4bcab-8809-59de-815f-e0d99311309a',
  '59263e4f-021c-5bbf-8cde-6d1f1c3dd406',
  '726c2fe3-5608-5d6a-aad0-aaed494e4e1e',
  '2389fde0-c2e1-535d-a909-ba7e0b1aacb1',
  '45186858-dd6f-595c-af22-ce4eda3ab7b6',
  '335c7af8-81f8-5fdd-8146-08ee17861680',
  '7f535f14-3faf-57d4-bff2-45effbadf1db',
  '29c0473b-46f9-5273-bab6-5f5b56dbcfc0'],
 'overlap_record_ids': ['e05576b6-795d-57c2-8896-96bb2562f148',
  'c3189663-9488-5843-a713-0adece2b7c47',

El filtro no es una operación decorativa aplicada al final de la consulta. Cambia el conjunto sobre el que debe calcularse el ranking: buscamos los vecinos más próximos entre los productos Einhell, no los productos Einhell que hayan sobrevivido por casualidad al top-10 global.

Por eso la comprobación no termina al ver que todos los resultados pertenecen a la marca correcta. También compararemos sus IDs con el oráculo exacto construido sobre ese mismo subconjunto. Solo así podremos distinguir un filtro funcional de una búsqueda condicionada que ha perdido vecinos durante la recuperación.


<a id="s03-qdrant-operaciones-crud"></a>

## 7. Operaciones CRUD

El canary test utilizará un UUID conocido y reservado para esta sesión. Sobre ese único registro recorreremos el ciclo completo: lo insertaremos, comprobaremos que puede recuperarse por ID y mediante búsqueda, modificaremos uno de sus campos y, finalmente, lo eliminaremos.

El objetivo no es solo confirmar que el motor admite operaciones CRUD. También queremos observar cuánto tarda cada cambio en hacerse visible desde las distintas rutas de lectura. Por eso registraremos los intentos y el tiempo transcurrido hasta detectar la inserción, la actualización y el borrado.

Al terminar eliminaremos únicamente el registro temporal de prueba. No borraremos el namespace ni el índice completo, porque la prueba debe ser segura y no afectar al resto de los datos ingeridos.

Un resultado inmediato tampoco demuestra que el sistema sea fuertemente consistente en todos los casos. Solo describe lo ocurrido para esta operación, en esta ruta de lectura y durante esta ejecución concreta. La prueba aporta evidencia observable, pero no permite generalizar una garantía más amplia que la documentada por el proveedor.

In [10]:
started = time.perf_counter()
client.upsert(RESOURCE_NAME, wait=True, points=[models.PointStruct(id=TEMPORARY_TEST_ID, vector=television_vector, payload={"text": "registro temporal", "metadata": {"record_id": TEMPORARY_TEST_ID, "product_id": "S03-TEMPORARY-TEST", "vector_id": -1, "title": "Registro temporal de prueba", "brand": "S03", "color": "amarillo", "locale": "es"}})])
fetched = client.retrieve(RESOURCE_NAME, ids=[TEMPORARY_TEST_ID], with_payload=True)
assert fetched
upsert_visible_s, upsert_attempts = time.perf_counter() - started, 1
client.set_payload(RESOURCE_NAME, payload={"metadata": {**fetched[0].payload["metadata"], "brand": "S03-updated"}}, points=[TEMPORARY_TEST_ID], wait=True)
updated = client.retrieve(RESOURCE_NAME, ids=[TEMPORARY_TEST_ID], with_payload=True)
assert updated[0].payload["metadata"]["brand"] == "S03-updated"
update_visible_s, update_attempts = 0.0, 1
client.delete(RESOURCE_NAME, points_selector=[TEMPORARY_TEST_ID], wait=True)
assert not client.retrieve(RESOURCE_NAME, ids=[TEMPORARY_TEST_ID])
delete_visible_s, delete_attempts = 0.0, 1

mutation = {"upsert": True, "fetch_or_query": True, "update": True, "delete": True}
temporary_test_visibility = {
    "upsert_seconds": upsert_visible_s,
    "upsert_attempts": upsert_attempts,
    "update_seconds": update_visible_s,
    "update_attempts": update_attempts,
    "delete_seconds": delete_visible_s,
    "delete_attempts": delete_attempts,
}
temporary_test_visibility

{'upsert_seconds': 0.008949250000000575,
 'upsert_attempts': 1,
 'update_seconds': 0.0,
 'update_attempts': 1,
 'delete_seconds': 0.0,
 'delete_attempts': 1}

<a id="s03-qdrant-informe"></a>

## 8. Informe de ejecución

El informe final conservará la información necesaria para reconstruir e interpretar esta ejecución: versiones del cliente y del servicio, destino consultado, número de registros visibles, tiempos observados, scores nativos, IDs recuperados y resultado completo del canary test.

Los tiempos deben leerse como una descripción del experimento, no como un benchmark entre proveedores. No hemos controlado calentamiento, concurrencia, red, hardware ni carga de fondo, y los motores ni siquiera se ejecutan en infraestructuras comparables. Una diferencia de milisegundos no permite concluir qué solución sería más rápida en producción.

Sí podemos comparar aquello que mantuvimos constante: el contrato funcional, los filtros aplicados, la semántica de las respuestas y el `recall@10` frente al mismo oráculo exacto. Esa evidencia permite saber si cada motor respeta los datos, las condiciones y el espacio vectorial definidos para la práctica.

Persistir estos resultados no convierte una única ejecución en una verdad general. Su valor está en dejar una traza reproducible: qué se probó, bajo qué configuración y qué ocurrió exactamente.

In [11]:
run = ProviderRun(
    provider="qdrant",
    provider_version=str(provider_version),
    target="Docker HTTP localhost:6333",
    resource=RESOURCE_NAME,
    record_count=record_count,
    score_kind="similarity",
    higher_is_better=True,
    query_id=TELEVISOR_QUERY_ID,
    query_text=str(television_row["query_text"]),
    top_k=TOP_K,
    hits=native_hits,
    exact_record_ids=television_evaluation["exact_record_ids"],
    recall_at_k=float(television_evaluation["recall_at_k"]),
    filtered_hits=filtered_drill_hits,
    filtered_recall_at_k=float(filtered_evaluation["recall_at_k"]),
    durations_ms={"ingestion": ingestion_ms, "television_query": query_ms},
    mutation=mutation,
    visibility={"initial_count_seconds": visibility_seconds, "initial_count_attempts": visibility_attempts, **temporary_test_visibility},
    notes=["Los tiempos describen esta ejecución y no forman un ranking entre proveedores."],
)
report_path = write_provider_run(run)
print(f"Informe escrito en {report_path.relative_to(PROJECT_ROOT)}")

Informe escrito en .artifacts/provider_runs/qdrant.json


<a id="s03-qdrant-consideraciones"></a>

## 9. Consideraciones específicas de Qdrant

Qdrant representa los productos como puntos con un vector y un payload JSON. HNSW es su índice para vectores densos; lo que configuramos son sus parámetros y las estructuras auxiliares de payload, no una selección entre familias ANN distintas.

El índice de metadata.brand es especialmente importante en esta práctica. Un filtro correcto no debe convertirse en un postprocesado accidental de los diez vecinos globales. Queremos que el motor busque dentro del universo permitido y después comparar ese ranking con el oráculo exacto filtrado.


<a id="s03-qdrant-proximos-pasos"></a>

## 10. Próximos pasos

Cada motor ofrece capacidades que van más allá de la búsqueda densa utilizada en esta comparación: recuperación híbrida, vectores sparse, múltiples representaciones por documento, cuantización, reranking, inferencia integrada o mecanismos específicos de multitenancy.

No las incorporaremos todavía porque cambiarían la pregunta del experimento. Si un proveedor utilizara búsqueda híbrida y otro únicamente embeddings densos, una diferencia en los resultados ya no podría atribuirse con claridad al motor, al índice o a la estrategia de recuperación.

La siguiente ampliación debería comenzar siempre por un requisito medible. Por ejemplo, mejorar el recall de consultas con referencias exactas, reducir memoria o aislar tenants con una garantía concreta. A partir de ahí se añadirá una sola capacidad cada vez y se repetirá la evaluación, de modo que podamos observar qué mejora introduce y qué coste añade.

El notebook de LangChain reutilizará las colecciones ya creadas en Chroma y Qdrant. No repetirá la ingesta ni construirá una copia paralela de los datos. Si una capa de abstracción necesita duplicar toda la colección para poder conectarse, ya no estaría envolviendo el mismo sistema: estaría creando otro despliegue y alterando la comparación.

<a id="s03-qdrant-limpieza"></a>

## 11. Limpieza

Hasta este punto hemos creado una colección de Qdrant y hemos cargado en ella el catálogo de productos. La celda siguiente permite eliminar esa colección cuando quieras terminar la práctica o empezar otra vez desde cero. Al ejecutarla se borrarán los vectores, los metadatos y el índice asociados a esa colección.

Para evitar un borrado accidental, la operación está desactivada inicialmente. Si quieres activarla, escribe en el archivo .env una confirmación con este formato:

    S03_CONFIRM_CLEANUP=DELETE:<nombre-de-la-colección>

Después guarda el archivo, vuelve a ejecutar la celda inicial de configuración del notebook y ejecuta la celda de limpieza. Si el valor no coincide exactamente con el nombre de la colección cargada, el notebook no borrará nada.

Esta acción elimina los datos de la colección, pero mantiene el contenedor y el volumen de Docker. Es lo normal si quieres volver a ejecutar el laboratorio más adelante. Si también quieres eliminar el entorno local de Qdrant y sus datos persistidos, ejecuta en una terminal:

    docker compose -f deploy/qdrant/compose.yaml down --volumes

El último comando también borra el volumen de Docker. Úsalo solo si realmente quieres reiniciar el motor desde cero.


In [ ]:
confirmation = os.getenv("S03_CONFIRM_CLEANUP", "")
expected_confirmation = f"DELETE:{RESOURCE_NAME}"

if confirmation != expected_confirmation:
    print({
        "cleanup": "omitida",
        "motivo": "Define S03_CONFIRM_CLEANUP con la confirmación exacta para habilitarla.",
        "confirmacion_requerida": expected_confirmation,
    })
else:
    client.delete_collection(RESOURCE_NAME)
    print({"cleanup": "completada", "recurso_eliminado": RESOURCE_NAME})


**Ejecuta también la siguiente celda si quieres eliminar el contenedor de Docker, así como la imagen de Qdrant y el volumen persistente:**

In [ ]:
!docker compose -f ../deploy/qdrant/compose.yaml down --volumes --remove-orphans --rmi all

/Users/aruizart/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=21066) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


[+] down 0/1
 ⠋ Container bbdd-vectoriales-s03-weaviate Stopping                         0.1s
[+] down 0/1
 ⠙ Container bbdd-vectoriales-s03-weaviate Stopping                         0.2s
[+] down 0/1
 ⠹ Container bbdd-vectoriales-s03-weaviate Stopping                         0.3s
[+] down 0/1
 ⠸ Container bbdd-vectoriales-s03-weaviate Stopping                         0.4s
[+] down 1/1
 ✔ Container bbdd-vectoriales-s03-weaviate Removed                          0.5s
[+] down 3/4
 ✔ Container bbdd-vectoriales-s03-weaviate               Removed            0.5s
 ✔ Image cr.weaviate.io/semitechnologies/weaviate:1.38.2 Removed            0.0s
 ✔ Volume bbdd-vectoriales-s03-weaviate-data             Removed            0.0s
 ⠋ Network bbdd-vectoriales-s03-weaviate_default         Removing           0.1s
[+] down 3/4
 ✔ Container bbdd-vectoriales-s03-weaviate               Removed            0.5s
 ✔ Image cr.weaviate.io/semitechnologies/weaviate:1.38.2 Removed            0.0s
 ✔ Volume bbdd-vec